<div style="display: flex; justify-content: space-between; align-items: center;">
   <h1 style="margin: 0;">Sage Image Search Lab</h1>
   <a href="https://sagecontinuum.org" target="_blank">
      <img src="../images/sage_logo.jpeg" alt="Sage Logo" style="width: 140px; height: 50px;">
   </a>
</div>
author: Francisco Lozano, francisco.lozano@northwestern.edu

![Hero Image](../images/hero_image.png)
>[**Sage Image Search**](https://portal.sagecontinuum.org/labs/image-search) - Production deployment, try it out!

Let's take a look at *Sage Image Search* together! This lab will walk you through the process of building a simple image search application using the Sage Image Search stack.

**Prerequisites:** Python, basic ML familiarity, 1 GPU (recommended), a [Hugging Face](https://huggingface.co/) account, and an [access token](https://huggingface.co/docs/hub/en/security-tokens) (`HF_TOKEN`; `read` role is enough) to download Gemma 4. See [learning.md](../learning.md).

After completing the lab, you will be able to:

- Explain the Sage Image Search pipeline: captioning, embedding, indexing, hybrid search, reranking, and search UI
- Run a simplified version of the stack locally using Milvus Lite, CLIP, Gemma 4 E2B, and a Gradio search UI
- Map each lab component to the production system (Weaviate, Triton, NRP , weavloader, Gradio API, React portal)
- Evaluate retrieval quality with MRR and Success@K on SageBench queries

This lab was created for [**Sage Grande: Summer of AI 2026**](https://sagecontinuum.org/docs/events/2026-Sage-Summer-Hackathon)

### Companion videos

| Video | When to watch | Link |
|-------|---------------|------|
| **Sage Image Search** — system overview and production walkthrough | After **Architecture overview**, or before Steps 1–8 | [Watch on YouTube](https://youtu.be/hzfKL0smzFM) |
| **Image Search Benchmarking** — metrics, Sagebench, and evaluation | Before or after **Step 9: Mini evaluation** | [Watch on YouTube](https://youtu.be/NUEs7AeGk4I) |

You can complete the lab without watching either video; they reinforce concepts from the SAGE team.

## NDP setup checklist

<!-- 1. Confirm **User_Persistent_Storage** is your current folder
2. Select the Sage Image Search Lab workspace in the NDP Widget
3. Run **Install requirements.txt** from `docs/notebooks/` (re-run each session)
4. Set `HF_TOKEN` in the next cell (create at https://huggingface.co/docs/hub/en/security-tokens) -->

Full steps: [learning.md#ndp-workspace-setup](../learning.md#ndp-workspace-setup)

In [ ]:
import os
from pathlib import Path

# os.environ["HF_TOKEN"] = "hf_..."  # uncomment if needed; create at https://huggingface.co/docs/hub/en/security-tokens
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    print("WARNING: HF_TOKEN not set. Accept https://huggingface.co/google/gemma-4-E2B-it license, then create a read token:")
    print("  https://huggingface.co/docs/hub/en/security-tokens")
else:
    print("HF_TOKEN is set.")

Here we will set the repo root and the persistent directory.

- **repo root**: This is the root directory of the repository. This will allow us to find the components of the production stack.
- **persistent directory**: This is the directory where we will store the lab data.


In [ ]:
# set the repo root
def find_repo_root() -> Path:
    """
    This function finds the root directory of the repository.
    """
    for p in [Path.cwd(), Path.cwd().parent.parent, Path.cwd() / "sage-nrp-image-search"]:
        if (p / "weavloader").exists() and (p / "docs").exists():
            return p.resolve()
    return Path.cwd().resolve()

REPO_ROOT = find_repo_root()
print(f"Repo root: {REPO_ROOT}")

# set the persistent directory
persist = [p for p in Path.cwd().parents if "Persistent" in p.name]
PERSIST_DIR = (persist[0] if persist else Path.cwd()) / "sage_image_search_lab"
PERSIST_DIR.mkdir(parents=True, exist_ok=True)
print(f"Persist dir: {PERSIST_DIR}")

Let's now download the required python packages for this lab. You can also do it manually by running the following command in the terminal:

```
pip install -r requirements.txt
```

If you are on Google Colab, you can run the following command:

```
!git clone https://github.com/waggle-sensor/sage-nrp-image-search
!pip install -r /content/sage-nrp-image-search/docs/notebooks/requirements.txt
```
>NOTE: if the code block below fails, try running the commands above.


In [ ]:
import sys, subprocess
req = Path("requirements.txt")
if not req.exists():
    req = REPO_ROOT / "docs" / "notebooks" / "requirements.txt"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])

Now let's set the seed for reproducibility and the sample size, so we can get the same results every time we run the lab. We will also set the device to use for the lab and retrieve the caption prompt from the production stack.

In [ ]:
import gc, importlib.util, random, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image

# initialize the model config
def load_model_config(repo_root: Path):
    """Load caption_model_prompt and clip_alpha from weavloader/inference/model_config.py."""
    cfg = repo_root / "weavloader" / "inference" / "model_config.py"
    spec = importlib.util.spec_from_file_location("model_config", cfg)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.caption_model_prompt, mod.clip_alpha

# Set the seed for reproducibility and the sample size
SEED, SAMPLE_SIZE, QUERY_ALPHA, TOP_K = 42, 50, 0.4, 25
random.seed(SEED); np.random.seed(SEED)

# Set the device to use for the lab
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("WARNING: No GPU. Gemma captioning uses SageBench summary fallback.")
    print("Stop server and relaunch with 1 GPU for the full lab.")
else:
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# load the model config
CAPTION_PROMPT, CLIP_ALPHA = load_model_config(REPO_ROOT)
print("Loaded caption_model_prompt and clip_alpha from weavloader/inference/model_config.py")
print(f"clip_alpha={CLIP_ALPHA} (image weight in fused CLIP vectors, same as production)")

## Load SageBench subset

Now we will load the SageBench subset. This is the dataset that we will use as our "Sage data stream".

![Image Sample](https://huggingface.co/datasets/sagecontinuum/SageBench/resolve/main/summary/random_image_sample.png)
>The SageBench repo on the Hub includes a notebook with a EDA analysis. **[SageBench EDA (Hub)](https://huggingface.co/datasets/sagecontinuum/SageBench/blob/main/summary/SageBench_eda_analysis.ipynb)**

Below is the difference between the production and lab implementation.
- **Production:** weavloader ingests live SAGE camera images via the [Python Sage Data Client](https://sagecontinuum.org/docs/tutorials/accessing-data).
- **Lab:** [sagecontinuum/SageBench](https://huggingface.co/datasets/sagecontinuum/SageBench) train split — `SAMPLE_SIZE=50`, `SEED=42`. View the link for more details on the dataset.

In [ ]:
from datasets import load_dataset
from io import BytesIO

# helper function to convert SageBench/HF image values to RGB PIL for matplotlib and Gradio
def to_pil_image(img) -> Image.Image:
    """Normalize SageBench/HF image values to an in-memory RGB PIL image."""
    if isinstance(img, Image.Image):
        pil = img.convert("RGB")
    elif isinstance(img, dict):
        if img.get("bytes"):
            pil = Image.open(BytesIO(img["bytes"])).convert("RGB")
        elif img.get("path"):
            pil = Image.open(img["path"]).convert("RGB")
        else:
            raise TypeError(f"Unsupported image dict keys: {list(img.keys())}")
    elif isinstance(img, str):
        pil = Image.open(img).convert("RGB")
    else:
        raise TypeError(f"Expected image, got {type(img)}")
    # Copy pixels into memory. Path-backed PIL images keep HF filenames that
    # Gradio tries to serve in the browser (and fails), showing image_id text instead.
    return Image.fromarray(np.asarray(pil, dtype=np.uint8))

# load the SageBench dataset
ds = load_dataset("sagecontinuum/SageBench", split="train")

# shuffle the dataset and select the first `SAMPLE_SIZE` rows
rows = list(ds.shuffle(seed=SEED).select(range(min(SAMPLE_SIZE, len(ds)))))
print(f"Loaded {len(rows)} rows")

# create a dictionary to store the seen images
seen, image_records = {}, []
for row in rows:
    iid = row["image_id"]
    if iid not in seen:
        seen[iid] = len(image_records)
        rec = dict(row)
        rec["image"] = to_pil_image(rec["image"])
        image_records.append(rec)

# create a dataframe of the queries with relevant images
query_df = pd.DataFrame([{
    "query_id": r["query_id"], "query_text": r["query_text"],
    "image_id": r["image_id"], "relevant": int(r["relevance_label"]),
} for r in rows if int(r["relevance_label"]) == 1])

# display the number of unique images, labeled queries, and the first image
print(f"Unique images: {len(image_records)} | Labeled queries: {len(query_df)}")
image_records[0]["image"]

## Architecture overview

> **Optional:** [Sage Image Search video walkthrough](https://youtu.be/hzfKL0smzFM) covers this pipeline on the live production stack.

Lets take a look at the architecture of the production and lab implementations. This will help us understand the differences and similarities between the two and the core components of the workflow.

![Production and Lab Architecture](../images/learning_arch.png)

| Component | Production | Lab |
|-----------|------------|-----|
| Image Data| Sage Stream (Python client) | SageBench (HF dataset) |
| Vector DB | Weaviate | Milvus Lite |
| Embedding Model | Triton CLIP-DFN5B, fused image+caption (`clip_alpha=0.7`) | open_clip ViT-B-32, same fusion |
| Captioning Model | `run_nrp_model()` → gemma-4-31B QAT via [NRP](https://nrp.ai/documentation/userdocs/ai/llm-managed/) | gemma-4-E2B-it via transformers |
| Search API | Gradio (`app/main.py`) | Gradio (notebook code below) |
| User interface | [React app](https://portal.sagecontinuum.org/labs/image-search) | Gradio (UI + API in one) |
| Search | [Weaviate](https://weaviate.io/) hybrid α=0.4 | [Milvus](https://milvus.io/) `hybrid_search` + `WeightedRanker` |

- **Production:** the [Sage Image Search portal](https://portal.sagecontinuum.org/labs/image-search) is a React web app. It calls the Gradio service in `app/` for hybrid search. 
- **Lab:** you build the same Gradio pattern inline, wired to Milvus `hybrid_search` + CrossEncoder rerank.

## Step 1: Embeddings

Here we will create the functions that will create embeddings for the `Image` from our dataset, the `Caption` that the VLM will generate, and `Search Text` the user will submit. 

![Learning Embeddings](../images/learning_embeddings.png)

Below is the difference between the production and lab implementation.
- **Production:** `get_clip_embeddings()` in [`weavloader/inference/model.py`](../../weavloader/inference/model.py) embeds **both** the VLM generated caption and the image via Triton CLIP, then fuses them into one vector with `clip_alpha` (default `0.7` = 70% image, 30% caption).

- **Lab:** same fusion math with `open_clip` ViT-B-32. Indexed vectors are created **after** captioning (Step 2).

In [ ]:
import open_clip

# Initialize the open_clip model and tokenizer
CLIP_MODEL, _, CLIP_PREPROCESS = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
CLIP_MODEL = CLIP_MODEL.to(DEVICE).eval()
CLIP_TOKENIZER = open_clip.get_tokenizer("ViT-B-32")

# Initialize the embedding functions
@torch.inference_mode()
def embed_image(image: Image.Image) -> np.ndarray:
    """Encode and L2-normalize an image (CLIP image tower)."""
    t = CLIP_PREPROCESS(image).unsqueeze(0).to(DEVICE)
    v = CLIP_MODEL.encode_image(t)
    return (v / v.norm(dim=-1, keepdim=True)).cpu().numpy().flatten()

@torch.inference_mode()
def embed_text(text: str) -> np.ndarray:
    """Encode and L2-normalize text (CLIP text tower)."""
    tokens = CLIP_TOKENIZER([text]).to(DEVICE)
    v = CLIP_MODEL.encode_text(tokens)
    return (v / v.norm(dim=-1, keepdim=True)).cpu().numpy().flatten()

def fuse_embeddings(img_emb: np.ndarray, txt_emb: np.ndarray, alpha: float = CLIP_ALPHA) -> np.ndarray:
    """Same fusion as weavloader/inference/model.py — weighted sum, re-normalized."""
    if img_emb.shape != txt_emb.shape:
        raise ValueError("img_emb and txt_emb must have the same dimension")
    combined = alpha * img_emb + (1.0 - alpha) * txt_emb
    norm = np.linalg.norm(combined)
    if norm == 0.0:
        return txt_emb.copy()
    return (combined / norm).astype(np.float32)

def get_clip_embeddings(text: str, image: Image.Image | None = None) -> np.ndarray:
    """Lab version of get_clip_embeddings(): fuse image+caption when indexing; text-only for queries."""
    txt_emb = embed_text(text)
    if image is None:
        return txt_emb
    img_emb = embed_image(image)
    return fuse_embeddings(img_emb, txt_emb, alpha=CLIP_ALPHA)

# Quick sanity check on one record (uses SageBench summary as stand-in until Gemma captions in Step 2)
_demo = image_records[0]
_demo_fused = get_clip_embeddings(_demo.get("summary") or "outdoor scene", _demo["image"])
_demo_text_only = get_clip_embeddings("outdoor scene", image=None)
print(f"Fused vector dim: {_demo_fused.shape}, text-only query dim: {_demo_text_only.shape}")
print(f"Fusion uses clip_alpha={CLIP_ALPHA} (production default)")

## Step 2: Captions (Gemma 4 E2B-it)

Here we will create our captioning function. We will also test what we built so far.

![Learning Captioning](../images/learning_captioning.png)

Below is the difference between the production and lab implementation.
- **Production:** `run_nrp_model()` → [gemma-4-31B-it-qat-w4a16-ct](https://huggingface.co/google/gemma-4-31B-it-qat-w4a16-ct). Production uses [NRP](https://nrp.ai/documentation/userdocs/ai/llm-managed/), they host different llms for users to use.
- **Lab:** [gemma-4-E2B-it](https://huggingface.co/google/gemma-4-E2B) with the **same** `caption_model_prompt`.

After captioning, production calls `get_clip_embeddings(caption, image)` to build the indexed vector. The lab does the same in the next cell.

> The [NRP](https://nrp.ai/) is a community-owned research and education platform connecting researchers and educators to foster collaboration, accelerate innovation, and share resources. NRP provides access to cutting-edge technologies in AI, high-performance computing, data storage, and networking.

In [ ]:
import gc
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig

GEMMA_MODEL_ID = "google/gemma-4-E2B-it"

# Initialize the captioning function
def caption_with_gemma(image: Image.Image, prompt: str, model, processor) -> str:
    """
    Captioning with Gemma 4 E2B-it using pre-loaded model instances.
    """
    # set up inputs
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": prompt},
    ]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, return_dict=True, return_tensors="pt",
        add_generation_prompt=True, enable_thinking=False, # Disabling thinking saves massive VRAM/Time
    )
    # Move inputs to the correct device
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[-1]
    
    # generate caption
    with torch.inference_mode():
        outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False)
        
    text = processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return text

# Handle configuration and loop logic with 4-bit optimization
USE_SUMMARY=False # if your GPU is not powerful enough for the VLM, set to True
if DEVICE == "cuda" and USE_SUMMARY != True:
    print("Loading Gemma 4 E2B-it in 4-bit NF4 precision...")
    
    # Configure 4-bit quantization
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    
    processor = AutoProcessor.from_pretrained(GEMMA_MODEL_ID)
    
    # Load the model with the quantization config
    model = AutoModelForMultimodalLM.from_pretrained(
        GEMMA_MODEL_ID, 
        quantization_config=quantization_config,
        device_map="auto"
    )
    
    print("Captioning with quantized Gemma 4 E2B-it...")
    for rec in image_records:
        rec["caption"] = caption_with_gemma(rec["image"], CAPTION_PROMPT, model, processor)
        
    # Clean up memory completely after the loop
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("CPU fallback: using SageBench summary as caption")
    for rec in image_records:
        rec["caption"] = rec.get("summary") or ""


# Lets Demo the embedding process now that we have captions
# ---------------------------------------------

## Print first 3 captions
for rec in image_records[:3]:
    print("---", rec["image_id"])
    print("Caption:", rec["caption"][:200], "...")

## Fuse CLIP embeddings 
print("Building fused image+caption vectors...")
for rec in image_records:
    rec["vector"] = get_clip_embeddings(rec["caption"], rec["image"])

## Get first query from dataset (if any) then embed it
DEMO_QUERY = query_df.iloc[0]["query_text"] if len(query_df) else "snow-covered lake from vsn W06C"
qvec = get_clip_embeddings(DEMO_QUERY, image=None)  # query path: text only

## Get top-5 images for query
sims = np.array([float(np.dot(qvec, rec["vector"])) for rec in image_records])
top_idx = np.argsort(sims)[::-1][:5]
print(f"Vector top-5 for query (text-only embedding): {DEMO_QUERY}")
for i in top_idx:
    print(f"  {image_records[i]['image_id']} sim={sims[i]:.3f}")

## Get similarity heatmap for query
n_show = min(12, len(image_records))
order = np.argsort(sims)[::-1][:n_show]
fig, ax = plt.subplots(figsize=(10, 2))
ax.imshow(sims[order].reshape(1, -1), aspect="auto", cmap="viridis")
ax.set_xticks(range(n_show))
ax.set_xticklabels([image_records[i]["image_id"][:8] for i in order], rotation=45, ha="right", fontsize=8)
ax.set_yticks([])
ax.set_title(f"Fused-vector similarity heatmap: {DEMO_QUERY[:50]}")
plt.tight_layout(); plt.show()

>NOTE: if your GPU is not powerful enough for the VLM, set `USE_SUMMARY=True`

## Step 3: Index

Here we build the Milvus Lite collection that stores fused CLIP vectors and searchable text for hybrid retrieval.

![Learning VectorDB](../images/learning_vectorDB.png)

Below is the difference between the production and lab implementation.
- **Production:** [Weaviate](https://weaviate.io/) stores the **fused** CLIP vector plus searchable caption/metadata text.
- **Lab:** [Milvus Lite](https://milvus.io/) with a dense `vector` field and built-in BM25 on `search_text` (sparse vectors generated automatically via `FunctionType.BM25`).
>NOTE: Weaviate and Milvus Lite are both vector databases (vectorDBs), but have different syntax, so the code is very different from each other. The production deployment is scheduled to migrate to Milvus in the future.

Run the code cell below, then read the breakdown that follows it.

In [ ]:
from pymilvus import MilvusClient, DataType, Function, FunctionType

# Define a helper function to parse MilvusClient search hits
def _parse_hits(hits):
    """
    Convert MilvusClient search hits to lab result dicts.
    """
    out = []
    for h in hits:
        rec = id_to_rec[h["id"]]
        out.append({
            "id": h["id"],
            "image_id": rec["image_id"],
            "caption": rec["caption"],
            "vsn": rec.get("vsn", ""),
            "zone": rec.get("zone", ""),
            "camera": rec.get("camera", ""),
            "job": rec.get("job", ""),
            "address": rec.get("address", ""),
            "score": float(h.get("distance", 0)),
            "image": rec["image"],
        })
    return out

# Convert vectors to list and build search text
for rec in image_records:
    rec["vector"] = rec["vector"].tolist() if hasattr(rec["vector"], "tolist") else rec["vector"]
    meta = " ".join(str(rec.get(k, "")) for k in ["vsn", "zone", "camera", "job", "address"])
    rec["search_text"] = f"{rec['caption']} {meta}".strip()

# Initialize path of Milvus Lite database file, collection name, and vector dimension
MILVUS_PATH = str(PERSIST_DIR / "sage_lab.db")
COLLECTION = "sage_lab"
VECTOR_DIM = len(image_records[0]["vector"])

# Initialize Milvus Lite client and drop collection if it exists
client = MilvusClient(MILVUS_PATH)
if client.has_collection(COLLECTION):
    client.drop_collection(COLLECTION)

# Create schema with fixed fields
schema = client.create_schema(enable_dynamic_field=False)
schema.add_field("id", DataType.INT64, is_primary=True)
schema.add_field("vector", DataType.FLOAT_VECTOR, dim=VECTOR_DIM)
schema.add_field(
    "search_text", DataType.VARCHAR, max_length=65535, enable_analyzer=True,
)
schema.add_field("sparse", DataType.SPARSE_FLOAT_VECTOR)
schema.add_field("image_id", DataType.VARCHAR, max_length=256)
schema.add_field("caption", DataType.VARCHAR, max_length=5000)
schema.add_field("vsn", DataType.VARCHAR, max_length=200)
schema.add_field("zone", DataType.VARCHAR, max_length=200)
schema.add_field("camera", DataType.VARCHAR, max_length=200)
schema.add_field("job", DataType.VARCHAR, max_length=200)
schema.add_field("address", DataType.VARCHAR, max_length=200)

# Add BM25 function to search_text field
schema.add_function(Function(
    name="search_text_bm25",
    function_type=FunctionType.BM25,
    input_field_names=["search_text"],
    output_field_names=["sparse"],
))

# Prepare index parameters
index_params = client.prepare_index_params()
index_params.add_index(field_name="vector", index_type="AUTOINDEX", metric_type="IP")
index_params.add_index(
    field_name="sparse",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="BM25",
)

# Create collection with specified schema and index parameters
client.create_collection(
    collection_name=COLLECTION,
    schema=schema,
    index_params=index_params,
)

# Insert data into collection
client.insert(collection_name=COLLECTION, data=[{
    "id": i,
    "vector": rec["vector"],
    "search_text": rec["search_text"][:4000],
    "image_id": str(rec["image_id"]),
    "caption": rec["caption"][:2000],
    "vsn": str(rec.get("vsn", "")),
    "zone": str(rec.get("zone", "")),
    "camera": str(rec.get("camera", "")),
    "job": str(rec.get("job", "")),
    "address": str(rec.get("address", "")),
} for i, rec in enumerate(image_records)])

# Create a dictionary to map IDs to records
id_to_rec = {i: r for i, r in enumerate(image_records)}
print(f"Indexed {len(image_records)} fused vectors + BM25 text → {MILVUS_PATH}")

The code above is filled with Milvus Lite syntax that may be confusing for beginners, so let's break it down.
>NOTE: Ignore the `Method not implemented! in AllocTimestamp` error. It is a known issue with Milvus Lite.

### What is Milvus Lite?

[Milvus Lite](https://milvus.io/blog/introducing-milvus-lite.md) is an **embedded vector database** — a single file on disk (`sage_lab.db` in your persistent folder) that stores vectors and supports fast similarity search. No Kubernetes or separate server process; `MilvusClient(MILVUS_PATH)` opens that file like SQLite.

**Production mapping:** [Weaviate](https://weaviate.io/) plays the same role — it stores fused CLIP vectors and searchable text for hybrid retrieval. Although it is not embedded, it is a production-ready server that you need to deploy separately.

---

### a. Prepare each record

```python
rec["search_text"] = f"{rec['caption']} {meta}".strip()
```

Before indexing, we build one searchable text blob per image: **Gemma caption + SAGE metadata** (`vsn`, `zone`, `camera`, …). This mirrors what production BM25 searches over in Weaviate (`caption`, `vsn`, `camera`, etc.).

We only **insert** raw `search_text`; Milvus generates the sparse BM25 representation for us.

---

### b. Schema — what fields exist?

| Field | Type | Purpose |
|-------|------|---------|
| `id` | `INT64` (primary key) | Row ID; we use `0 … N-1` to match `image_records` |
| `vector` | `FLOAT_VECTOR` | Fused CLIP embedding from Step 2 (dense semantic search) |
| `search_text` | `VARCHAR` + analyzer | Raw caption + metadata for keyword search |
| `sparse` | `SPARSE_FLOAT_VECTOR` | **Auto-filled by Milvus** from `search_text` via BM25 — you never set this manually |
| `image_id`, `caption`, `vsn`, … | `VARCHAR` | Metadata returned in search results |

`enable_analyzer=True` on `search_text` tells Milvus to tokenize text (split into words) for BM25, similar to how Weaviate analyzes text fields.

---

### c. BM25 function — automatic keyword vectors

```python
schema.add_function(Function(
    name="search_text_bm25",
    function_type=FunctionType.BM25,
    input_field_names=["search_text"],
    output_field_names=["sparse"],
))
```

This is Milvus's built-in **full-text search** hook:

- **Input:** `search_text` (plain string you insert)
- **Output:** `sparse` (BM25-weighted sparse vector, computed on insert)

You do not need manual tokenization — Milvus handles it at index time.

---

### d. Indexes — how search stays fast

**What is an index?**

An index is a **pre-built lookup structure** the database creates when you insert data. Instead of scanning every row on each query (a *full scan*), the index organizes data so Milvus can jump straight to the most likely matches.

Think of the difference like this:

- **No index:** Read every image record one by one and compute similarity — fine for 50 lab images, too slow for millions of SAGE photos.
- **With index:** Use a structure built at insert time (like a book's index or a search-engine inverted list) to retrieve top candidates in milliseconds.

We pay a small cost at **insert time** (building the index) to make **search time** much faster. Production Weaviate builds the same kind of structures on its vector and inverted-index fields.

**Why it matters in this lab**

| Field | Index | Metric | Used for |
|-------|-------|--------|----------|
| `vector` | `AUTOINDEX` | `IP` (inner product) | Approximate nearest-neighbor (ANN) on fused CLIP vectors |
| `sparse` | `SPARSE_INVERTED_INDEX` | `BM25` | Keyword / full-text search on `search_text` |

- **`AUTOINDEX` on `vector`** — Milvus decides which graph-based ANN index to use (e.g. HNSW-style) so dense search does not compare your query against every stored embedding. With ~50 vectors the speedup is invisible; at SAGE scale it is essential.
- **`SPARSE_INVERTED_INDEX` on `sparse`** — Classic inverted index: terms like `W097` or `Hawaii` map to the rows that contain them, so BM25 scoring runs on a short candidate list instead of every caption.

**IP vs cosine:** Our CLIP vectors are L2-normalized, so **inner product (IP)** and **cosine similarity** rank results the same way. We use `IP` here because it avoids an extra normalization step at query time making it faster.

---

### e. Insert

```python
client.insert(collection_name=COLLECTION, data=[{...}])
```

Each row stores the fused `vector` we calculated, the `search_text` string, and metadata fields. Milvus runs the BM25 function and populates `sparse` automatically.

---

### f. Helpers for later steps

- **`id_to_rec`** — maps Milvus row `id` back to the in-memory `image_records` entry (so we can show PIL images in results).
- **`_parse_hits()`** — converts Milvus search output into the dict format used by vector, keyword, hybrid, and Gradio search below.

---

**Try it:** After running the cell, confirm `Indexed N fused vectors + BM25 text → …/sage_lab.db`. That file persists in `User_Persistent_Storage` — re-running later steps does not require re-captioning if you skip re-indexing.

## Step 4: Vector search

Now let's break down *Hybrid Search*. First, we will take a look at the vector search (semantic search) part. Here will define our vector search function and take a look at the results.

![Learning Vector Search](../images/learning_vectorsearch.png)

Dense ANN only — no keywords.
>NOTE: The `vector_search()` function is only to explain the concept of vector search. In our workflow, we use the `hybrid_search()` function to perform vector search.

In [ ]:
# initialize the vector search function
def vector_search(query: str, limit: int = TOP_K):
    """Dense ANN on fused CLIP vectors (text-only query embedding)."""
    hits = client.search(
        collection_name=COLLECTION,
        data=[get_clip_embeddings(query, image=None).tolist()],
        anns_field="vector",
        search_params={"metric_type": "IP"},
        limit=limit,
        output_fields=["image_id", "caption", "vsn", "zone", "camera", "job", "address"],
    )
    return _parse_hits(hits[0])

# run the vector search function on a demo query and print the results
vec_results = vector_search(DEMO_QUERY)
for r in vec_results[:5]:
    print(f"  {r['image_id']} vsn={r['vsn']} score={r['score']:.3f}")

Let's plot the results.

In [ ]:
# plot the results
fig, axes = plt.subplots(1, min(3, len(vec_results)), figsize=(12, 4))
axes = [axes] if len(vec_results) == 1 else axes
for ax, r in zip(axes, vec_results[:3]):
    ax.imshow(r["image"]); ax.set_title(f"{r['vsn']}\n{r['caption'][:50]}..."); ax.axis("off")
plt.suptitle(f"Vector search: {DEMO_QUERY[:60]}"); plt.tight_layout(); plt.show()

## Step 5: Keyword search

Now that we understand the vector search part, let's break down the keyword search part in *Hybrid Search*. Here we will define our keyword search function and take a look at the results.

![Learning Keyword Search](../images/learning_keywordsearch.png)

>NOTE: The `keyword_search()` function is only to explain the concept of keyword search. In our workflow, we use the `hybrid_search()` function to perform keyword search.

In [ ]:
# lets define a meta query to use for keyword search
META_QUERY = "W097 bottom_camera Hawaii" if any("W097" in str(r.get("vsn","")) for r in image_records) else DEMO_QUERY

# initialize the keyword search function
def keyword_search(query: str, limit: int = TOP_K):
    """BM25 keyword search via Milvus sparse field."""
    hits = client.search(
        collection_name=COLLECTION,
        data=[query],
        anns_field="sparse",
        search_params={"metric_type": "BM25"},
        limit=limit,
        output_fields=["image_id", "caption", "vsn", "zone", "camera", "job", "address"],
    )
    return _parse_hits(hits[0])

# run the keyword search function on the meta query and print the results
kw_results = keyword_search(META_QUERY)
print(f"Keyword search: {META_QUERY}")
for r in kw_results[:5]:
    print(f"  {r['image_id']} vsn={r['vsn']} bm25={r['score']:.3f}")

## Step 6: Hybrid search

Now that we understand both vector and keyword search, let's put them together in *Hybrid Search*. Here we will define our hybrid search function and take a look at the results.

![Learning Hybrid Search](../images/learning_hybridsearch.png)

**Production:** `clip_hybrid_query()` with `query_alpha=0.4` in `app/HyperParameters.py` (40% vector, 60% keyword).

**Lab:** Milvus `hybrid_search()` with `WeightedRanker(QUERY_ALPHA, 1 - QUERY_ALPHA)` over dense + BM25 paths — same fusion concept as Weaviate hybrid.

In [ ]:
from pymilvus import AnnSearchRequest, WeightedRanker

def hybrid_search(query: str, alpha: float = QUERY_ALPHA, limit: int = TOP_K):
    """Milvus native hybrid: fused CLIP vector search + BM25, ranked with WeightedRanker."""
    dense_req = AnnSearchRequest(
        data=[get_clip_embeddings(query, image=None).tolist()],
        anns_field="vector",
        param={"metric_type": "IP"},
        limit=limit,
    )
    sparse_req = AnnSearchRequest(
        data=[query],
        anns_field="sparse",
        param={"metric_type": "BM25"},
        limit=limit,
    )
    hits = client.hybrid_search(
        collection_name=COLLECTION,
        reqs=[dense_req, sparse_req],
        ranker=WeightedRanker(alpha, 1.0 - alpha),
        limit=limit,
        output_fields=["image_id", "caption", "vsn", "zone", "camera", "job", "address"],
    )
    return _parse_hits(hits[0])

results = hybrid_search(DEMO_QUERY)
print(f"Hybrid (alpha={QUERY_ALPHA}): {DEMO_QUERY}")
for r in results[:5]:
    print(f"  {r['image_id']} vsn={r['vsn']} hybrid_score={r['score']:.3f}")

The code above is filled with Milvus Lite syntax that may be confusing for beginners, so let's break it down.

### What is hybrid search?

**Hybrid search** runs **two retrievers** on the same query and **merges** their ranked lists into one:

1. **Dense (vector)** — "find images *semantically* like this text" (CLIP embedding similarity).
2. **Sparse (BM25)** — "find images whose captions/metadata *contain these words*" (keyword match on `vsn`, `Hawaii`, `camera`, etc.).

Neither alone is perfect. Vector search handles `"snowy mountain landscape"` well but may miss an exact `W097`. Keyword search nails `W097 bottom_camera` but struggles with vague scene descriptions. **Hybrid combines both.**

**Production mapping:** `clip_hybrid_query()` in `app/query.py` does the same fusion in Weaviate with `query_alpha=0.4` (40% vector, 60% keyword). Our lab uses Milvus `hybrid_search()` + `WeightedRanker(0.4, 0.6)` for the same idea.

---

### a. Two `AnnSearchRequest` objects — one per search path

`AnnSearchRequest` packages **one** approximate-nearest-neighbor (ANN) query against **one** vector field. Hybrid search needs **two** requests because we search **two different fields**:

| Request | Field searched | Query input | What it finds |
|---------|----------------|-------------|---------------|
| `dense_req` | `vector` | CLIP embedding of your text (`get_clip_embeddings`) | Semantically similar images |
| `sparse_req` | `sparse` | Raw query string | BM25 keyword matches on `search_text` |

Each request returns its own top-`limit` candidates. Milvus does **not** merge them yet — that happens in the next step.

```python
dense_req = AnnSearchRequest(
    data=[get_clip_embeddings(query, image=None).tolist()],
    anns_field="vector",
    param={"metric_type": "IP"},
    limit=limit,
)
sparse_req = AnnSearchRequest(
    data=[query],              # plain text — Milvus tokenizes for BM25
    anns_field="sparse",
    param={"metric_type": "BM25"},
    limit=limit,
)
```

**Why text-only CLIP for the query?** Same as production: at search time you type words, not an image. `get_clip_embeddings(query, image=None)` embeds the query text only.

---

### b. `client.hybrid_search()` — run both paths together

```python
hits = client.hybrid_search(
    collection_name=COLLECTION,
    reqs=[dense_req, sparse_req],
    ranker=WeightedRanker(alpha, 1.0 - alpha),
    limit=limit,
    output_fields=[...],
)
```

Milvus runs **both** ANN searches, then passes both result lists to the **ranker**. The final `limit` rows are the fused ranking — not simply "top 5 from vector + top 5 from keyword."

---

### c. `WeightedRanker` — blending scores with `alpha`

```python
WeightedRanker(QUERY_ALPHA, 1.0 - QUERY_ALPHA)   # default: WeightedRanker(0.4, 0.6)
```

| Weight | Path | Meaning |
|--------|------|---------|
| `0.4` (`alpha`) | Dense / vector | 40% influence from semantic similarity |
| `0.6` (`1 - alpha`) | Sparse / BM25 | 60% influence from keyword relevance |

This matches `query_alpha=0.4` in `app/HyperParameters.py`:

> *An alpha of 1 is pure vector search. An alpha of 0 is pure keyword search.*

Milvus normalizes scores from each path (they use different scales) before applying the weights, so a high BM25 score and a high IP score can be compared fairly.

**Try it:** Change `QUERY_ALPHA` at the top of the notebook (e.g. `0.0`, `0.4`, `1.0`) and re-run this cell. At `1.0` results should resemble Step 4; at `0.0`, Step 5.

---

### d. `_parse_hits()` — same helper as before

Hybrid search returns the same hit structure as vector and keyword search. `_parse_hits()` attaches the PIL `image` and metadata from `id_to_rec` so later steps (reranking, Gradio, evaluation) all use one result format.

---

### e. How this fits the full pipeline

```
Query text
    ├─► dense_req  → vector ANN  ─┐
    └─► sparse_req → BM25 ANN   ─┼─► WeightedRanker(0.4, 0.6) → top-K results
```

Steps 4 and 5 isolated each path for learning. **Step 6 onward** (reranking, Gradio, evaluation) always calls `hybrid_search()` — that is what production uses too, before the cross-encoder reranker in Step 7.

## Step 7: Reranking

Now that we have our hybrid search function, let's add a reranking step to improve the results for our users. Here we will define our reranking function and take a look at the results.

![Learning Reranker](../images/learning_reranker.png)

**Production:** `ms-marco-MiniLM-L-6-v2` via Weaviate reranker.

**Lab:** same CrossEncoder via `sentence-transformers`.

In [ ]:
from sentence_transformers import CrossEncoder

# initialize the reranker
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE)

# initialize the reranking function
def rerank(query: str, candidates, limit: int = TOP_K):
    """
    Rerank candidates with CrossEncoder.
    """
    scores = reranker.predict([[query, c["caption"]] for c in candidates])
    out = [dict(c, rerank_score=float(s)) for c, s in zip(candidates, scores)]
    out.sort(key=lambda x: x["rerank_score"], reverse=True)
    return out[:limit]

# run the reranking function on the demo query and print the results
reranked = rerank(DEMO_QUERY, results)
print("Before → after reranking (top 3):")
for b, a in zip(results[:3], reranked[:3]):
    print(f"  {b['image_id']} hybrid={b['score']:.3f} → rerank={a['rerank_score']:.3f}")

## Step 8: User Interface

We finally have the full pipeline! Let's build a user interface to search for images using our pipeline.

![Search Examples](../images/search_examples.jpeg)

**Production:** [`app/main.py`](../../app/main.py) runs a Gradio API server (`text_query()` → `clip_hybrid_query()`). Users search via the [React app at portal.sagecontinuum.org/labs/image-search](https://portal.sagecontinuum.org/labs/image-search).

**Lab:** build the same Gradio interface in this notebook, wired to `hybrid_search()` + `rerank()` instead of Weaviate. Students can read every line — no imports from `app/`.

In [ ]:
import gradio as gr
import os

os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"

def lab_text_query(description: str):
    """
    Lab version of app/main.py text_query() — hybrid search + rerank,
    return (image, caption) tuples for the gallery + metadata table.
    """
    if not description or not description.strip():
        return [], gr.DataFrame(value=None)

    ranked = rerank(description.strip(), hybrid_search(description.strip(), limit=TOP_K), limit=TOP_K)

    # Return (PIL.Image, caption) tuples — same pattern as app/main.py line 172.
    images = []
    rows = []
    for r in ranked:
        rec = id_to_rec[r["id"]]
        images.append((to_pil_image(rec["image"]), str(r["image_id"])))
        rows.append({
            "image_id": r["image_id"],
            "caption": r["caption"],
            "score": round(r["score"], 4),
            "hybrid_score": round(r["score"], 4),
            "rerank_score": round(r.get("rerank_score", 0), 4),
            "vsn": rec.get("vsn", ""),
            "camera": rec.get("camera", ""),
            "zone": rec.get("zone", ""),
            "address": rec.get("address", ""),
        })
    return images, pd.DataFrame(rows)

def build_gradio_ui():
    """
    Build the Gradio UI for the Sage Image Search Lab.
    Mirrors app/main.py load_interface() layout (Text Query tab).
    """
    with gr.Blocks(title="Sage Image Search Lab") as demo:
        gr.Markdown(
            """
            # Text Query
            Enter a text description to find similar images.

            **Production UI:** [portal.sagecontinuum.org/labs/image-search](https://portal.sagecontinuum.org/labs/image-search) (React app → Gradio API)
            """
        )
        query = gr.Textbox(label="Text Query", interactive=True)
        examples = gr.Dataset(
            label="Example Queries",
            components=[query],
            samples=[
                ["Show me images in Hawaii"],
                ["Rainy Chicago"],
                ["Snowy Mountains"],
                ["Show me clouds in the top camera"],
                ["Cars in W049"],
                ["intersection in the right camera"],
            ],
        )
        with gr.Row():
            submit_btn = gr.Button("Submit")
            clear_btn = gr.Button("Clear")
        gr.Markdown("### Returned Images")
        gallery = gr.Gallery(label="Returned Images", columns=[3], object_fit="contain", height="auto")
        meta = gr.DataFrame(label="Metadata", show_copy_button=True)

        def clear():
            return "", [], gr.DataFrame(value=None)

        def on_select(evt: gr.SelectData):
            return evt.value[0]

        submit_btn.click(fn=lab_text_query, inputs=query, outputs=[gallery, meta])
        clear_btn.click(fn=clear, outputs=[query, gallery, meta])
        examples.select(fn=on_select, outputs=query)
    return demo

# build the Gradio UI
lab_demo = build_gradio_ui()
lab_demo.launch(server_name="0.0.0.0", share=True, show_error=True)

## Step 9: Mini evaluation

> **Optional:** [Image Search Benchmarking video](https://youtu.be/NUEs7AeGk4I) explains the benchmarking tooling.



**How good is our lab pipeline at finding the right images?** We measure that with the same primary metrics production uses — on a small SageBench slice instead of the full benchmark suite.

See [METRICS.md](../../benchmarking/benchmarks/METRICS.md) for the full evaluation strategy; [Sagebench](../../benchmarking/benchmarks/Sagebench/Readme.md) for the production benchmark harness; and [benchmarking.md](../benchmarking.md) for more information on the benchmarking suite.

![Learning Benchmarking](../images/learning_benchmarking.png)
>  To find out more checkout our [blog]((https://sagecontinuum.org/labs/image-search-bench)) about our Benchmarking Tooling.

### Metrics (K = `TOP_K` = 25)

Production always evaluates at **K = `response_limit` (25)** — the number of results the system actually returns. We use the same fixed K here ([METRICS.md](../../benchmarking/benchmarks/METRICS.md#1%EF%B8%8F-keep-k-fixed)).

| Metric | What it measures | Lab formula | Production column (`query_eval_metrics.csv`) |
|--------|------------------|-------------|---------------------------------------------|
| **MRR** (Mean Reciprocal Rank) | How *early* the first relevant image appears. `1.0` = relevant image ranked #1; `0.5` = first hit at rank 2; `0.0` = no hit in top-K. | Mean of `1/rank` per query (or 0) | `rerank_score_reciprocal_rank` → mean = MRR |
| **Success@25** (Hit Rate@25) | Did the user get *at least one* relevant image in the top 25? `1` = yes, `0` = no. | Mean of binary hit per query | `hit` → mean = Success@25 |

These are the **primary model-selection metrics** in production (50/50 weight on the leaderboards). Checkout the [Sagebench leaderboard](../../benchmarking/benchmarks/overall/leaderboard.ipynb). There is also a [leaderboard](../../benchmarking/benchmarks/overall/leaderboard.ipynb) that takes into account all benchmarks.

**Supporting metrics** (Precision@25, NDCG@25, Recall@25, Diversity@25) are reported in full Sagebench runs but skipped in this mini lab to keep the notebook short.

### Production vs lab

| | **Production (Sagebench)** | **Lab (this notebook)** |
|---|---------------------------|-------------------------|
| **Harness** | `imsearch_eval` → `benchmarking/benchmarks/Sagebench/run_benchmark.py` | Inline `evaluate_query()` loop |
| **Dataset** | `sagecontinuum/SageBench` (`SAMPLE_SIZE`, `SEED` env vars) | Same dataset; 50 rows, `SEED=42` |
| **Labels** | Binary `relevance_label` per query–image pair | Same (`query_df`) |
| **K** | `RESPONSE_LIMIT=25` | `TOP_K=25` |
| **Queries evaluated** | All labeled queries in the run | All labeled queries in our subset (small N — treat scores as directional, not publishable) |
| **Outputs** | `query_eval_metrics.csv`, `image_search_results.csv`, `evaluate.ipynb` | Printed MRR and Success@25 |
| **Run locally** | `cd benchmarking/benchmarks/Sagebench && make run-local SAMPLE_SIZE=50` | This cell |

**Caveat:** With only ~50 dataset rows and few dozen queries, lab scores have high variance. Production Sagebench on the full dataset is required for serious comparisons — see [METRICS.md](../../benchmarking/benchmarks/METRICS.md#4%EF%B8%8F-interpretation-guidance) on sample sizes for detecting improvements.

Run the code cell below, then compare your numbers to a Sagebench report under `benchmarking/benchmarks/Sagebench/results/`.

In [ ]:
# Build relevance sets: all image_ids labeled relevant for each query_id
relevant_by_query = query_df.groupby("query_id")["image_id"].apply(set).to_dict()

# initialize the evaluation function
def evaluate_query(query_text, relevant_ids, k=TOP_K):
    """
    Evaluation function for a single query.
    Returns MRR and hit (Success@K).oooi
    """
    ranked = rerank(query_text, hybrid_search(query_text, limit=k), limit=k)
    retrieved = [r["image_id"] for r in ranked]
    for rank, rid in enumerate(retrieved, 1):
        if rid in relevant_ids:
            return 1.0 / rank, 1
    return 0.0, 0

# Evaluate every unique labeled query in our SageBench subset
eval_queries = query_df.drop_duplicates("query_id")
mrrs, hits = [], []
for _, q in eval_queries.iterrows():
    mrr, hit = evaluate_query(q["query_text"], relevant_by_query[q["query_id"]])
    mrrs.append(mrr)
    hits.append(hit)

# calculate the mean of the mrrs and hits then print the results
mrr = float(np.mean(mrrs))
success_at_k = float(np.mean(hits))
print(f"Queries evaluated: {len(mrrs)} (K={TOP_K})")
print(f"MRR: {mrr:.4f}")
print(f"Success@{TOP_K}: {success_at_k:.4f}")
print()
print("Production reports the same primary metrics from query_eval_metrics.csv:")
print("  MRR          = mean(rerank_score_reciprocal_rank)")
print(f"  Success@{TOP_K} = mean(hit)")

## Conclusion

![Workflow](../images/workflow.jpeg)

You just walked through the full **Sage Image Search** retrieval pipeline — the same stages production uses to index SAGE edge-camera images and answer natural-language queries.

### End-to-end workflow

| Stage | What you built (lab) | What production uses |
|-------|----------------------|----------------------|
| **1. Data** | SageBench subset (50 images, seed 42) | Live SAGE camera stream via `weavloader` |
| **2. Caption** | Gemma 4 E2B-it + `caption_model_prompt` | NRP `run_nrp_model()` → Gemma 4 31B QAT |
| **3. Embed** | Fused open_clip vector (`clip_alpha=0.7`) | Triton CLIP + same fusion in `get_clip_embeddings()` |
| **4. Index** | Milvus Lite (`vector` + BM25 on `search_text`) | Weaviate (`clip` vector + inverted index) |
| **5. Search** | `hybrid_search` + `WeightedRanker(0.4, 0.6)` | `clip_hybrid_query()` in `app/query.py` |
| **6. Rerank** | CrossEncoder `ms-marco-MiniLM-L-6-v2` | Weaviate `reranker-transformers` (same model family) |
| **7. UI** | Gradio in this notebook | [React portal](https://portal.sagecontinuum.org/labs/image-search) → Gradio API |
| **8. Evaluate** | Mini MRR & Success@25 on SageBench | Full [Sagebench](../../benchmarking/benchmarks/Sagebench/) via `imsearch_eval` |

### Key ideas to take away

1. **Captions bridge pixels and words** — VLM captions make images searchable by text; metadata (`vsn`, `camera`, `zone`) strengthens keyword matching.
2. **Hybrid beats either alone** — Vector search understands scene semantics; BM25 nails exact node IDs and metadata terms. Production blends them with `alpha=0.4`.
3. **Reranking sharpens the top of the list** — Hybrid retrieval casts a wide net; the cross-encoder re-scores query–caption pairs so the best matches rise to the top.
4. **Same metrics, different scale** — Your lab MRR / Success@25 use the same definitions as [METRICS.md](../../benchmarking/benchmarks/METRICS.md); run `make run-local` in Sagebench for production-grade numbers.

The diagram above is the map. Each **Step** in this notebook is one stop on that path — simplified for learning, but aligned with the system running at [portal.sagecontinuum.org/labs/image-search](https://portal.sagecontinuum.org/labs/image-search).


## Stretch & choose your path

Now that you've seen the system, you can choose your own adventure!

![Learning Adventure](../images/learning_adventure.png)

| Level | Next | Difficulty |
|-------|------|------------|
| Explorer | [overview.md](../overview.md), [glossary.md](../glossary.md) | Easy |
| Tinkerer | [configuration.md](../configuration.md) | Medium |
| Builder | [getting-started.md](../getting-started.md), [benchmarking.md](../benchmarking.md) | Hard |
| Contributor | [CONTRIBUTING.md](../CONTRIBUTING.md) | Hard |
| Pioneer | [Image Search at the Edge](https://sagecontinuum.org/docs/events/2026-Sage-Summer-Hackathon#potential-hackathon-projects) | Very Hard |



Thank you for your time and effort! I hope you enjoyed the lab. If you have any questions, please reach out to me at francisco.lozano@northwestern.edu or submit an issue on [GitHub](https://github.com/waggle-sensor/sage-nrp-image-search/issues).

---

<div style="display: flex; flex-direction: column; gap: 20px;">
   
   <!-- Top Row: NSF Logo and Text Side-by-Side (Left-Aligned) -->
   <div style="display: flex; align-items: center; gap: 15px; width: 100%;">
      <a href="https://nsf.gov" target="_blank" style="flex-shrink: 0;">
         <img src="../images/nsf_logo.png" alt="NSF Logo" style="width: 50px; height: 50px; display: block;">
      </a>
      <p style="margin: 0; flex: 1; text-align: left;">
         This material is based upon work supported by the National Science Foundation under Grant Nos. 1935984, 2331263, and 2436842.
         Any opinions, findings, and conclusions or recommendations expressed in this material are those of the author(s) and do not necessarily reflect the views of the National Science Foundation.
      </p>
   </div>

   <!-- Bottom Row: Other Logos Only (Right-Aligned Underneath) -->
   <div style="display: flex; align-items: center; gap: 20px; justify-content: flex-end; width: 100%;">
      <a href="https://naise.northwestern.edu/" target="_blank">
         <img src="../images/nu_logo.png" alt="Northwestern Logo" style="width: 150px; height: 50px; display: block;">
      </a>
      <a href="https://www.anl.gov/" target="_blank">
         <img src="../images/argonne_logo.png" alt="Argonne Logo" style="width: 160px; height: 60px; display: block;">
      </a>
   </div>

</div>



